# Hydrogen atom orbitals

*quantum · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/hydrogen_radial)

Writing $\psi = (u(r)/r)\,Y_{lm}(\theta, \phi)$ turns the 3D hydrogen problem into a 1D radial equation with an extra centrifugal term $l(l+1)/2r^2$. One finite-difference Hamiltonian per angular momentum $l$ is stacked into a $(L, N, N)$ tensor and a single batched torch.linalg.eigh returns every orbital at once. The animation cycles through 1s, 2s, 2p, 3s, … showing the radial probability and a slice through $\lvert\psi_{nl0}\rvert^2$.

**Vector form**

$$-\tfrac12\,u''(r) + \Big(\frac{l(l+1)}{2r^2} - \frac1r\Big)\,u(r) = E\,u(r),\qquad \psi_{nlm}(\mathbf r) = \frac{u_{nl}(r)}{r}\,Y_{lm}(\theta,\varphi)$$

**Index (tensor) form**

$$H_{lij} = -\tfrac12\,D2_{ij} + V_{li}\,\delta_{ij},\quad V_{li} = \frac{l(l+1)}{2r_i^2} - \frac1{r_i},\qquad H_{lij}\,u_{ljn} = E_{ln}\,u_{lin}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $r_i$ | `radial_grid(N, r_max)` | (N,) | interior points $r = dr \dots r_{\max} - dr$ ($u = 0$ at both ends) |
| $V_{li}$ | `effective_potential(r, ls)` | (L, N) | Coulomb attraction plus centrifugal barrier, one row per $l$ |
| $H_{lij}$ | `hamiltonian(D2, V)` | (L, N, N) | a stack of $L$ Hamiltonians, one per angular momentum |
| $E_{ln},\ u_{lin}$ | `torch.linalg.eigh(H)` | (L, N), (L, N, N) | batched over $l$: energies and radial functions; principal number $n = l + 1 + \text{column}$ |
| $P(r) = u^2$ | `radial_probability(u)` | (N,) | probability per unit radius, $r^2\lvert R\rvert^2 = u^2$ |
| $\vert \psi_{nl0}\vert ^2$ | `orbital_density(R, l, cos_theta)` | (P, P) | $\lvert R(r)\,Y_{l0}(\theta)\rvert^2$ on a slice through the $z$-axis |

Atomic units ($\hbar = m_e = e = 4\pi\varepsilon_0 = 1$): lengths in Bohr radii, energies in hartree. The exact energies $E_n = -1/(2n^2)$ depend only on $n$, so 2s and 2p (or 3s, 3p, 3d) come out degenerate even though they are eigenvalues of different matrices — the hidden symmetry of the $1/r$ potential. The radial function of $(n, l)$ has $n - l - 1$ nodes. The grid stops at $r_{\max}$, so states whose size ($\approx 2n^2$) approaches $r_{\max}$ are squeezed and pushed up in energy; finite differences also slightly underbind 1s near the $1/r$ singularity. The slice shows $m = 0$ orbitals in the $xz$-plane ($z$ up), $Y_{l0} \propto P_l(\cos\theta)$; colour uses a compressed power-law scale $(\lvert\psi\rvert^2)^{0.3}$ so faint outer lobes stay visible.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/hydrogen_radial/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import effective_potential, hamiltonian, orbital_density, radial_probability

SPECTROSCOPIC = "spdf"


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n_max = params["n_max"]
    r, dr = tensors.radial_grid(params["n_grid"], params["r_max"], dev)
    ls = tensors.angular_momenta(n_max, dev)  # (L,)
    H = hamiltonian(tensors.second_derivative(len(r), dr, dev), effective_potential(r, ls))  # (L, N, N)

    energies, u = solve.eigenstates(H, dr)

    frames = []
    for n, ell in solve.orbitals(n_max):
        E, u_nl = solve.pick(energies, u, n, ell)
        half = 3.0 * n**2 + 3.0  # slice size follows the orbital size ~ n²
        r_slice, cos_theta = tensors.slice_grid(half, 161, dev)
        density = orbital_density(solve.radial_on_slice(u_nl, dr, r_slice), ell, cos_theta)  # (P, P)
        nodes = solve.count_nodes(u_nl)
        name = f"{n}{SPECTROSCOPIC[ell]}"
        print(f"{name}:  E = {E:9.5f}   exact -1/(2n^2) = {-0.5 / n**2:9.5f}   nodes = {nodes} (n-l-1 = {n - ell - 1})")
        frames.append(
            dict(name=name, n=n, l=ell, E=E, nodes=nodes, P=radial_probability(u_nl), density=density, half=half)
        )
    return plot.animate(r, frames, out_dir / "hydrogen_radial.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/hydrogen_radial/equation.py
r"""Physics → tensor translation.

The hydrogen atom in atomic units, -\tfrac12 \nabla^2 \psi - \psi / r = E \psi, separates as
\psi_{nlm} = R_{nl}(r) Y_{lm}(\theta, \varphi). With u(r) = r R(r) the radial part is a 1D problem

    -\tfrac12 u'' + V_l(r) u = E u,      V_l(r) = \frac{l(l+1)}{2 r^2} - \frac{1}{r},      u(0) = u(\infty) = 0.

On N radial grid points, and for L values of l at once, it is a stack of matrix eigenproblems:

    V_{li} = \frac{l(l+1)}{2 r_i^2} - \frac{1}{r_i}
    H_{lij} = -\tfrac12 D2_{ij} + V_{li} \delta_{ij}
    H_{lij} u_{ljn} = E_{ln} u_{lin}                     (j summed; l is a batch index, never summed)

The radial probability is P(r) = r^2 |R|^2 = u^2, and for m = 0 the angular part is
Y_{l0}(\theta) = \sqrt{(2l+1)/4\pi} \, P_l(\cos\theta) with the Legendre polynomial P_l.
"""

import math

import torch


def effective_potential(r: torch.Tensor, ls: torch.Tensor) -> torch.Tensor:
    r"""V_{li} = l(l+1)/(2 r_i^2) - 1/r_i.   r: (N,), ls: (L,) → (L, N)"""
    centrifugal = torch.einsum("l,i->li", ls * (ls + 1) / 2, 1 / r**2)  # (L, N) outer product
    return centrifugal - 1 / r  # Coulomb part broadcast over l


def hamiltonian(D2: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    r"""H_{lij} = -\tfrac12 D2_{ij} + V_{li} \delta_{ij}.   D2: (N, N), V: (L, N) → (L, N, N)"""
    return -0.5 * D2 + torch.diag_embed(V)  # diag_embed puts each row V_l on the diagonal of its own matrix


def radial_probability(u: torch.Tensor) -> torch.Tensor:
    r"""P(r) = r^2 |R(r)|^2 = u(r)^2.   (..., N) → (..., N)"""
    return u**2


def spherical_harmonic_m0(ell: int, cos_theta: torch.Tensor) -> torch.Tensor:
    r"""Y_{l0} = \sqrt{(2l+1)/4\pi} P_l(\cos\theta).   cos_theta: any shape → same shape"""
    return math.sqrt((2 * ell + 1) / (4 * math.pi)) * torch.special.legendre_polynomial_p(cos_theta, ell)


def orbital_density(R: torch.Tensor, ell: int, cos_theta: torch.Tensor) -> torch.Tensor:
    r"""|\psi_{nl0}|^2 = |R_{nl}(r) Y_{l0}(\theta)|^2.   R, cos_theta: (P, P) → (P, P)"""
    return (R * spherical_harmonic_m0(ell, cos_theta)) ** 2


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/hydrogen_radial/tensors.py
"""Build the concrete tensors: radial grid, second-derivative matrix, the l values and a 2D slice grid."""

import torch


def radial_grid(n: int, r_max: float, device: torch.device) -> tuple[torch.Tensor, float]:
    """N interior points r_i = i·dr, i = 1..N, dr = r_max/(N+1); u = 0 at r = 0 and r = r_max. Returns (r (N,), dr)."""
    dr = r_max / (n + 1)
    r = dr * torch.arange(1, n + 1, dtype=torch.float64, device=device)  # (N,)
    return r, dr


def second_derivative(n: int, dr: float, device: torch.device) -> torch.Tensor:
    """D2_{ij}: −2 on the diagonal, 1 on the two neighbouring diagonals, all over dr². Shape (N, N)."""
    main = -2.0 * torch.ones(n, dtype=torch.float64, device=device)  # (N,)
    side = torch.ones(n - 1, dtype=torch.float64, device=device)  # (N-1,)
    return (torch.diag(main) + torch.diag(side, 1) + torch.diag(side, -1)) / dr**2


def angular_momenta(n_max: int, device: torch.device) -> torch.Tensor:
    """l = 0, 1, …, n_max − 1: every l that appears in shells up to n_max. Shape (L,)."""
    return torch.arange(n_max, dtype=torch.float64, device=device)


def slice_grid(half_width: float, n_pix: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Points of the xz-plane in [−a, a]²: returns (r (P, P), cos θ (P, P)) with θ measured from +z."""
    s = torch.linspace(-half_width, half_width, n_pix, dtype=torch.float64, device=device)  # (P,)
    Z, X = torch.meshgrid(s, s, indexing="ij")  # rows: z, columns: x
    r = torch.sqrt(X**2 + Z**2).clamp_min(1e-9)
    return r, Z / r


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/hydrogen_radial/solve.py
"""Diagonalise all L radial Hamiltonians in one batched eigh, then pick out the orbitals (n, l)."""

import math

import torch


def eigenstates(H: torch.Tensor, dr: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (E_{ln} (L, N), u_{lin} (L, N, N)) with ∫ u² dr = 1 and u > 0 just outside the nucleus.

    eigh works on the last two axes and loops over the leading batch axis l by itself.
    """
    energies, u = torch.linalg.eigh(H)  # (L, N), (L, N, N)
    u = u / math.sqrt(dr)
    sign = torch.sign(u[:, :1, :])  # (L, 1, N) sign of each state at the first grid point
    return energies, u * torch.where(sign == 0, 1.0, sign)


def orbitals(n_max: int) -> list[tuple[int, int]]:
    """All (n, l) with n ≤ n_max, in the order 1s, 2s, 2p, 3s, 3p, 3d, …"""
    return [(n, ell) for n in range(1, n_max + 1) for ell in range(n)]


def pick(energies: torch.Tensor, u: torch.Tensor, n: int, ell: int) -> tuple[float, torch.Tensor]:
    """Energy and u(r) of orbital (n, l): column n − l − 1 of block l (radial quantum number)."""
    column = n - ell - 1
    return energies[ell, column].item(), u[ell, :, column]


def count_nodes(u: torch.Tensor, tol: float = 1e-4) -> int:
    """Sign changes of u(r), ignoring the tiny noisy tail where |u| < tol · max|u|."""
    big = u[u.abs() > tol * u.abs().max()]
    return int((big[1:] * big[:-1] < 0).sum())


def radial_on_slice(u: torch.Tensor, dr: float, r: torch.Tensor) -> torch.Tensor:
    """R = u/r evaluated at arbitrary radii r (any shape) by linear interpolation of the grid values."""
    u0 = torch.cat([u.new_zeros(1), u])  # (N+1,) add u(0) = 0, so u0[i] sits at r = i·dr
    position = (r / dr).clamp(0, len(u) - 0.001)  # fractional index
    i0 = position.floor().long()
    w = position - i0
    u_r = (1 - w) * u0[i0] + w * u0[i0 + 1]
    return u_r / r


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/hydrogen_radial/plot.py
"""Cycle through the orbitals: radial probability u² (left) and a slice through |ψ_nl0|² (right)."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.colors import PowerNorm

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(r: torch.Tensor, orbitals: list[dict], path: Path, hold: int = 15, draw_in: int = 6) -> Path:
    """Each orbital is shown for `hold` frames; its radial curve is drawn in over the first `draw_in`."""
    rs = r.cpu().numpy()
    n_max = max(o["n"] for o in orbitals)
    r_view = 3.0 * n_max**2 + 5.0

    fig, (ax, ax_img) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1.15, 1]})
    ax.set_xlim(0, r_view)
    ax.set_xlabel("r  (Bohr radii)")
    ax.set_ylabel("radial probability  r²|R|² = u²")
    (line,) = ax.plot([], [], color=ACCENT[0], lw=2)
    fill = [ax.fill_between(rs, 0 * rs, color=ACCENT[0], alpha=0.2)]
    label = ax.text(0.97, 0.95, "", transform=ax.transAxes, va="top", ha="right", family="monospace", fontsize=9)

    first = orbitals[0]
    image = ax_img.imshow(first["density"].cpu().numpy(), origin="lower", cmap="magma", interpolation="bilinear")
    ax_img.set_xlabel("x")
    ax_img.set_ylabel("z")
    ax_img.grid(False)
    title = ax_img.set_title("")

    def update(step: int):
        orbital = orbitals[min(step // hold, len(orbitals) - 1)]
        P = orbital["P"].cpu().numpy()
        shown = min(1.0, (step % hold + 1) / draw_in)  # fraction of the curve drawn so far
        n_drawn = max(2, int(shown * (rs < r_view).sum()))
        line.set_data(rs[:n_drawn], P[:n_drawn])
        fill[0].remove()
        fill[0] = ax.fill_between(rs[:n_drawn], P[:n_drawn], color=ACCENT[0], alpha=0.2)
        ax.set_ylim(0, 1.15 * P.max())
        n, ell, E = orbital["n"], orbital["l"], orbital["E"]
        label.set_text(
            f"{orbital['name']}  (n={n}, l={ell})\nE = {E:.4f}\n-1/2n² = {-0.5 / n**2:.4f}\nnodes = {orbital['nodes']}"
        )

        density = orbital["density"].cpu().numpy()
        a = orbital["half"]
        image.set_data(density)
        image.set_extent((-a, a, -a, a))
        image.set_norm(PowerNorm(0.3, vmin=0, vmax=density.max()))  # compressed colour scale shows faint lobes
        title.set_text(f"|ψ_{n}{ell}0|² in the xz-plane")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(hold * len(orbitals)), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_max': 3,
    'r_max': 60.0,
    'n_grid': 1500
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.hydrogen_radial")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.hydrogen_radial.main")
problem_dir = Path("problems/quantum/hydrogen_radial")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
